# Lab 1 — Linear Regression from Scratch
**Machine Learning Sup'Com**

Code skeletons for the lab statement. Fill in every `# TODO`. Each task ends with a *Check yourself* cell: compare your values with the statement before moving on.

Rules: the model uses **NumPy only** (pandas for data, matplotlib for plots, scikit-learn only to *check* your work). Prices are in thousand TND. **The test set stays closed until Task 13.**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParmas["figure.figsize"] = (6,4)
np.set_printoptions(precision=4, suppress=True)

AttributeError: module 'matplotlib.pyplot' has no attribute 'rcParmas'

## Part A — Data
### Task 1 — Load, sanity check and split

In [14]:
df = pd.read_csv("dataset.csv") # dataframe
df.shape  # number of rows and columns : tuple
df.dtypes # data type of each column : serie
df.describe() # statistical summary table : dataframe
df.isna().sum() # number of missing values per column : series
df.duplicated().sum() # number of duplicated rows : int
df["neighborhood"].value_counts() # frequencie of each neighberhood
# TODO: sanity checks. Write down what you find in a markdown cell below.
# df.shape, df.dtypes, df.describe() (read min and max!), df.isna().sum(),
# df.duplicated().sum(), df["neighborhood"].value_counts()
print(df.shape)

(110, 30)


*Your observations:* ...

In [190]:
print("Initial:", len(df))

# 1. Normalize neighborhood
df["neighborhood"] = df["neighborhood"].str.strip().str.capitalize()

# 2. Count duplicates
print("Duplicate rows:", df.duplicated().sum())

df_no_duplicates = df.drop_duplicates()
print("After duplicates:", len(df_no_duplicates))

# 3. Count invalid rows
invalid = (
    (df_no_duplicates["area"] <= 0) |
    (df_no_duplicates["area"] > 1000) |
    (df_no_duplicates["age"] < 0) |
    (df_no_duplicates["price"] <= 0)
)

print("Invalid rows:", invalid.sum())

df_clean = df_no_duplicates[~invalid].reset_index(drop=True)
print("Final:", len(df_clean))

Initial: 110
Duplicate rows: 5
After duplicates: 105
Invalid rows: 5
Final: 100


In [191]:
# TODO: split 60 / 20 / 20 with the given seed
m = len(df_clean)
idx = np.random.default_rng().permutation(m)
print(idx[:5])            # expected [90 92  7 14 47]
print(m)
train = idx[:int(0.6*m)]   # first 60 shuffled indices
val   = idx[int(0.6*m):int(0.8*m)]  # next 20
test  = idx[int(0.8*m):]   # last 20

[ 9 93 73 57 75]
100


idx = np.random.default_rng(2).permutation(m) : create random index list wiyh memory

In [198]:
# TODO: impute 'age' and 'distance_center' with the TRAINING median (same numbers for val and test)
# Create the three datasets
df_clean_train = df_clean.iloc[train].copy()
df_clean_val = df_clean.iloc[val].copy()
df_clean_test = df_clean.iloc[test].copy()

# Calculate medians ONLY from training data
median_age = df_clean_train["age"].median()
median_distance_center = df_clean_train["distance_center"].median()

# Impute missing values using the same training medians
for col in ["age", "distance_center"]:
    median = median_age if col == "age" else median_distance_center

    df_clean_train[col] = df_clean_train[col].fillna(median)
    df_clean_val[col] = df_clean_val[col].fillna(median)
    df_clean_test[col] = df_clean_test[col].fillna(median)

print(df_clean_train.shape, df_clean_val.shape, df_clean_test.shape)

print(train.shape, val.shape, test.shape)   # (60, 30) (20, 30) (20, 30)

(60, 30) (20, 30) (20, 30)
(60,) (20,) (20,)


 df_clean_train[col] = df_clean_train[col].fillna(median) : fillna fill the empty space
 df_clean_train = df_clean.iloc[train].copy() : select only the rows within the train list


**Check yourself:** raw shape (110, 30); 5 duplicated rows; 8 missing `age`, 6 missing `distance_center`; 10 spellings → 4; 5 impossible rows removed → 100 rows; indices `[90 92 7 14 47]`; training medians age = 19.0, distance_center = 13.7.

### Task 2 — Explore the training set and prepare the features

In [ ]:
# TODO: plots on the TRAINING set only
# (1) histogram of price
# (2) scatter: price vs area, distance_center, age
# (3) box plot of price per neighborhood
# (4) correlation heat map of the numeric features + bar chart of corr with price

In [199]:
print("X has NaN:", np.isnan(X_train).any())
print("y has NaN:", np.isnan(y_train).any())
print("X has infinity:", np.isinf(X_train).any())
print("y has infinity:", np.isinf(y_train).any())

X has NaN: area                    False
living_area             False
built_area              False
usable_area             False
rooms                   False
bathrooms               False
floor                   False
age                      True
distance_center          True
distance_metro          False
distance_highway        False
has_garage              False
listing_views           False
photos                  False
days_on_market          False
has_pool                False
has_elevator            False
sea_view                False
has_garden              False
has_ac                  False
has_heating             False
has_terrace             False
has_alarm               False
furnished               False
has_balcony             False
has_parking             False
has_concierge           False
neighborhood_Lac        False
neighborhood_Manouba    False
neighborhood_Marsa      False
dtype: bool
y has NaN: False
X has infinity: area                    False
living_area   

*Written justification (a few sentences):* is price–area close to a line? Does the spread grow with the area? Does the neighbourhood shift prices? What would make us reject a linear model?

...

In [193]:
# TODO: drop area_ft2, one-hot encode neighborhood (drop 'Ariana'),
# learn the columns on train and reuse them for val and test.
# Dummy columns must come at the END, in the order Lac, Manouba, Marsa.
# Separate features and target


# Drop area_ft2 and the target
df_features = df_clean.drop(columns=["area_ft2", target])

# One-hot encode neighborhood
X = pd.get_dummies(
    df_features,
    columns=["neighborhood"],
    drop_first=True,
    dtype=int
)

# Put dummy columns at the end in the required order
dummy_cols = [
    "neighborhood_Lac",
    "neighborhood_Manouba",
    "neighborhood_Marsa"
]
other_cols = [c for c in X.columns if c not in dummy_cols]
X = X[other_cols + dummy_cols]

feature_names = X.columns.tolist()

# Recreate train, validation, and test
X_train, y_train = X.iloc[train], df_clean[target].iloc[train]
X_val, y_val = X.iloc[val], df_clean[target].iloc[val]
X_test, y_test = X.iloc[test], df_clean[target].iloc[test]

print(X_train.shape, X_val.shape, X_test.shape)   # (60, 30) (20, 30) (20, 30)

(60, 30) (20, 30) (20, 30)


**Check yourself:** corr(area, area_ft2) = 1.00; area with living/built/usable_area ≈ 0.995; area with rooms 0.87; price with area 0.85; shapes (60,30), (20,30), (20,30).

## Part B — Build the model
### Task 3 — Prediction and cost

In [194]:
def predict(X, w, b):
    y_hat = X @ w + b  # for all examples at once (no loop)
    # TODO
    return y_hat


def compute_cost(X, y, w, b):
    j=(np.linalg.norm(predict(X, w, b) - y)**2)/(2*X.shape[0])
    #"""J(w, b) = 1/(2m) * ||X w + b - y||^2"""
    # TODO
    return j

In [195]:
# Tiny example
Xt = np.array([[1., 2.], [3., 4.], [5., 6.]])
yt = np.array([3., 7., 10.])
wt, bt = np.array([1., 1.]), 0.0

print(predict(Xt, wt, bt))             # [ 3.  7. 11.]
print(compute_cost(Xt, yt, wt, bt))    # 0.1667

# Real data, raw features, w = 0, b = 0
print(compute_cost(X_train, y_train, np.zeros(X_train.shape[1]), 0.0))   # 71724.59

[ 3.  7. 11.]
0.16666666666666666
nan


### Task 4 — Gradients and one descent step
*On paper first:* derive $\partial J/\partial b$ and show $\nabla_w J = \frac1m X^\top r$, $\ \partial J/\partial b = \mathrm{mean}(r)$.

In [205]:
def compute_gradients(X, y, w, b):
    """Return (dw, db)."""
    # TODO
    m = X.shape[0]
    errors = predict(X, w, b)-y
    dw = (X.T @ errors) /m
    db = np.mean(errors)
    return dw, db
    raise NotImplementedError


def gradient_step(X, y, w, b, alpha):
    """One gradient descent step. Return the new (w, b)."""
    # TODO
    c=compute_gradients(X, y, w, b)
    dw, db = c[0], c[1]
    return (w-alpha*dw, b-alpha*db)
    raise NotImplementedError

In [ ]:
# ---- PROVIDED: numerical gradient (central differences), used to check compute_gradients
def numerical_gradient(X, y, w, b, lam=0.0, eps=1e-5):
    """Approximate (dw, db) with [J(p + eps) - J(p - eps)] / (2 eps).
    The argument lam is only used from Task 12 on (compute_cost must then accept it)."""
    def cost(w_, b_):
        return compute_cost(X, y, w_, b_) if lam == 0 else compute_cost(X, y, w_, b_, lam)

    dw = np.zeros_like(w, dtype=float)
    for j in range(len(w)):
        e = np.zeros_like(w, dtype=float); e[j] = eps
        dw[j] = (cost(w + e, b) - cost(w - e, b)) / (2 * eps)
    db = (cost(w, b + eps) - cost(w, b - eps)) / (2 * eps)
    return dw, db

In [207]:
# Tiny example
dw, db = compute_gradients(Xt, yt, wt, bt)
print(dw, db)                                   # [1.6667 2.    ] 0.3333
print(gradient_step(Xt, yt, wt, bt, 0.1))       # (array([0.8333, 0.8   ]), -0.0333)

# Real data: one step from w = 0, b = 0 with alpha = 1e-6
n = X_train.shape[1]
w0, b0 = np.zeros(n), 0.0
w1, b1 = gradient_step(X_train, y_train, w0, b0, 1e-6)
print(compute_cost(X_train, y_train, w1, b1))   # 48349.36
print(compute_gradients(X_train, y_train, w0, b0)[1])   # -350.755

# Numerical check at a random point (differences must be < 1e-5)
rng = np.random.default_rng(0)
wr, br = rng.normal(size=n), rng.normal()
dw_a, db_a = compute_gradients(X_train, y_train, wr, br)
dw_n, db_n = numerical_gradient(X_train, y_train, wr, br)
print(np.max(np.abs(dw_a - dw_n)), abs(db_a - db_n))


[1.6667 2.    ] 0.3333333333333333
(array([0.8333, 0.8   ]), np.float64(-0.03333333333333333))
nan
-303.7269230769231


NameError: name 'numerical_gradient' is not defined

### Task 5 — Gradient descent and the loss curve

In [ ]:
def gradient_descent(X, y, alpha, n_iters):
    """Start from w = 0, b = 0. Return w, b, history (cost after each update)."""
    # TODO
    raise NotImplementedError

In [ ]:
# TODO: raw training features, 1000 iterations, alpha in {1e-4, 1e-5, 1e-6}.
# Plot the three loss curves (log vertical axis). Which alphas work? Is the result satisfying?
import warnings
warnings.filterwarnings("ignore")   # overflow warnings are expected for alpha = 1e-4

**Check yourself:** α = 1e-4 explodes (nan); α = 1e-5 gives J = 2635.1; α = 1e-6 gives J = 2801.3. Best achievable: J* = 304.27.

### Task 6 — Feature scaling

In [ ]:
class StandardScaler:
    """z = (x - mean) / scale. If a column has std 0, use scale 1."""

    def fit(self, X):
        # TODO: store self.mean_ and self.scale_
        return self

    def transform(self, X):
        # TODO
        raise NotImplementedError

    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [ ]:
scaler = StandardScaler().fit(X_train)
Z_train = scaler.transform(X_train)
Z_val   = scaler.transform(X_val)
Z_test  = scaler.transform(X_test)

print(Z_train.mean(axis=0).round(6)[:5], Z_train.std(axis=0)[:5])
print(Z_train[0, :5])        # 0.66, 0.668, 0.705, 0.491, 0.874

In [ ]:
# ---- PROVIDED: true optimum by least squares (intercept column of ones added)
m_train = len(y_train)
theta_star = np.linalg.lstsq(np.c_[np.ones(m_train), Z_train], y_train, rcond=None)[0]
b_star, w_star = theta_star[0], theta_star[1:]
J_star = compute_cost(Z_train, y_train, w_star, b_star)
print(J_star)                # 304.27

In [ ]:
# TODO: gradient descent on the SCALED training data, 2000 iterations,
# alpha in {0.001, 0.01, 0.1, 0.3, 0.5}. Plot the curves and, for each alpha,
# print the first iteration with J <= 1.01 * J_star (or "never").

**Check yourself:** α = 0.5 diverges; α = 0.3 reaches 1.01 J* at iteration 616; α = 0.1 at 1846; α = 0.01 never (J = 320.4 after 2000); α = 0.001 ends at J = 1623.9.

### Task 7 — The `LinearRegression` class

In [ ]:
class LinearRegression:
    def __init__(self, alpha=0.1, n_iters=50000):
        # TODO: only store the hyperparameters
        pass

    def fit(self, X, y):
        # TODO: run gradient_descent; store coef_, intercept_, loss_history_; return self
        raise NotImplementedError

    def predict(self, X):
        # TODO
        raise NotImplementedError

In [ ]:
model = LinearRegression().fit(Z_train, y_train)
print(model.loss_history_[-1])                              # 304.2683
print(model.loss_history_[99], model.loss_history_[999])    # 332.59  308.55
print(model.intercept_, y_train.mean())                     # 350.755 (= mean of y_train)

from sklearn.linear_model import LinearRegression as SkLR
sk = SkLR().fit(Z_train, y_train)
print(np.max(np.abs(model.coef_ - sk.coef_)))                         # < 0.1
print(np.max(np.abs(model.predict(Z_val) - sk.predict(Z_val))))       # < 0.02

# TODO: plot model.loss_history_

### Task 8 — Metrics

In [ ]:
def mse(y_true, y_pred):
    raise NotImplementedError   # TODO

def rmse(y_true, y_pred):
    raise NotImplementedError   # TODO

def mae(y_true, y_pred):
    raise NotImplementedError   # TODO

def r2_score(y_true, y_pred):
    raise NotImplementedError   # TODO

In [ ]:
yt_, yp_ = np.array([3., 5., 7.]), np.array([2., 5., 9.])
print(mse(yt_, yp_), rmse(yt_, yp_), mae(yt_, yp_), r2_score(yt_, yp_))   # 1.6667 1.2910 1.0 0.375

# TODO: compare with sklearn.metrics, then evaluate the model of Task 7 on train and validation.
# Do NOT touch the test set.

**Check yourself:** train RMSE 24.67, MAE 16.69, R² 0.970; validation RMSE 52.04, MAE 39.09, R² 0.891.

## Part C — Selecting a model
### Task 9 — Cross-validation

In [ ]:
def k_fold_indices(m, k, seed):
    """Shuffle range(m) with np.random.default_rng(seed).permutation, cut with np.array_split.
    Return the list of the k index arrays (the validation part of each fold)."""
    # TODO
    raise NotImplementedError


def cross_val_rmse(X, y, k=5, seed=0, alpha=0.1, n_iters=50000):
    """Return the list of the k fold RMSEs.
    The scaler AND the model are created inside the loop, fitted on the training part of the fold only."""
    # TODO
    raise NotImplementedError

In [ ]:
folds = k_fold_indices(60, 5, 0)
print(sorted(folds[0]))     # [2, 4, 8, 10, 16, 20, 27, 34, 42, 51, 52, 57]

# Candidate models (column indices in feature_names)
cols_A = [feature_names.index("area")]
cols_B = [feature_names.index(c) for c in
          ["area", "bathrooms", "age", "distance_center", "floor", "has_garage",
           "neighborhood_Lac", "neighborhood_Manouba", "neighborhood_Marsa"]]
cols_C = list(range(len(feature_names)))

# TODO: for A, B, C report mean and std of the CV RMSE, next to the validation RMSE.

**Check yourself:** CV RMSE A 74.68 ± 14.83, B 34.55 ± 8.28, C 58.19 ± 11.48; validation RMSE A 77.65, B 53.48, C 52.04. Which one would you trust, and why?

### Task 10 — Generalization

In [ ]:
# TODO (1): training RMSE, validation RMSE and gap for A, B, C
# TODO (2): learning curves for B and C with the first m' in {30, 40, 50, 60} training rows
#           (refit the scaler each time); plot train RMSE (on those m' rows) and val RMSE vs m'
# TODO (3): coefficients of model C for area, living_area, built_area, usable_area, and their sum

**Check yourself:** train/val RMSE A 74.31/77.65, B 30.71/53.48, C 24.67/52.04. C's coefficients: 100.0, −14.2, 7.2, 30.4 (sum 123.4).

*Why does C nearly memorise the data when m' = 30?* ...

## Part D — Regularization
### Task 11 — Change the objective: ridge regression
Answer on paper (or here, in LaTeX):

1. Derive $\nabla_w J_\lambda$ and $\partial J_\lambda/\partial b$.
2. Show that the update can be written $w \leftarrow (1-\tfrac{\alpha\lambda}{m})\,w - \alpha\,\tfrac1m X^\top r$. Why is it called *weight decay*?
3. Why do we not penalise $b$? (two sentences)
4. Compute by hand $J_\lambda$ and the gradients of the tiny example with $\lambda=3$.

...

In [ ]:
# Optional: check your hand computation (after Task 12 is done)
# print(compute_cost(Xt, yt, wt, bt, 3.0))            # 1.1667
# print(compute_gradients(Xt, yt, wt, bt, 3.0))       # (array([2.6667, 3.    ]), 0.3333)

### Task 12 — The class with $\lambda$
Extend `compute_cost`, `compute_gradients`, `gradient_descent` and `LinearRegression` with an optional `lam` (default 0.0). The bias is **not** penalised. Rerun all earlier checks: they must still pass with `lam=0`.

In [ ]:
# TODO: redefine (or edit above) the four pieces with the extra argument lam=0.0
#   compute_cost(X, y, w, b, lam=0.0)
#   compute_gradients(X, y, w, b, lam=0.0)
#   gradient_descent(X, y, alpha, n_iters, lam=0.0)
#   LinearRegression(alpha=0.1, n_iters=50000, lam=0.0)

In [ ]:
# Check: gradient with lam, numerical vs analytic
# dw_a, db_a = compute_gradients(X_train, y_train, wr, br, 10.0)
# dw_n, db_n = numerical_gradient(X_train, y_train, wr, br, lam=10.0)

# Compare with scikit-learn's Ridge(alpha=10)
# from sklearn.linear_model import Ridge
# mine = LinearRegression(lam=10).fit(Z_train, y_train)
# sk_r = Ridge(alpha=10).fit(Z_train, y_train)
# print(np.max(np.abs(mine.coef_ - sk_r.coef_)))                    # < 1e-6
# print(np.linalg.norm(mine.coef_))                                 # 80.16 (138.07 with lam = 0)

### Task 13 — Experiments: with and without regularization

In [ ]:
lams = [0, 0.1, 1, 3, 10, 30, 100, 300]

# TODO: for each lambda, on the 30 standardised features:
#   - fit on the training set, compute train RMSE and validation RMSE
#   - 5-fold CV RMSE (cross_val_rmse with lam passed along; extend its signature)
# Build a table, plot the three RMSE against lambda (log axis; draw lambda = 0 at the left edge).

In [ ]:
# TODO (2): weight paths of area, living_area, built_area, usable_area against lambda
# TODO (3): lam_star = argmin of the CV RMSE

In [ ]:
# TODO (4): FIRST USE OF THE TEST SET. Models trained on the training set only:
#   - least squares, all 30 features (lam = 0)
#   - least squares, key features B
#   - ridge, all 30 features, lam = lam_star
# Report RMSE, MAE, R2 for each.

**Check yourself:** CV RMSE for λ = 0, 0.1, 1, 3, 10, 30, 100, 300: 58.19, 55.86, 50.92, 47.64, 47.31, 54.45, 72.91, 98.29 → λ* = 10. Validation RMSE 52.04 → 41.48. Training RMSE 24.67 → 29.44 (86.96 at λ = 300).

| model | RMSE | MAE | R² |
|---|---|---|---|
| least squares, all 30 | 58.54 | 40.84 | 0.810 |
| least squares, key features B | 49.64 | 35.49 | 0.864 |
| ridge, all 30, λ* = 10 | 46.52 | 36.61 | 0.880 |

### Task 14 — Wrap-up
Short answers (a few sentences each).

**1.** Why did we keep the test set closed until the very end? (Hint: the test RMSE alone would have been lowest at λ = 30, with 41.98.)

...

**2.** Why can we not choose between A, B, C, or choose λ, using the training error alone?

...

**3.** Ridge on all features beats the hand-picked features B on the test set, although B had the better CV RMSE (34.55 vs 47.31). Does this make feature selection useless? Why can a CV score on 60 examples mislead? When would you still select features by hand?

...

**4.** What would happen to ridge if the features were not standardised?

...

**5.** *(optional)* Why is the L1 penalty not differentiable at 0, and what would you use instead of the gradient there?

...